# LeetCode 766 — Toeplitz Matrix bằng Python

**Link đề:** https://leetcode.com/problems/toeplitz-matrix/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix (ma trận)

```python
class Solution:
    def isToeplitzMatrix(self, matrix: list[list[int]]) -> bool:
        ...
```

## Mục tiêu học
1. Đọc và duyệt **ma trận 2 chiều** (list lồng list) bằng chỉ số `matrix[i][j]`.
2. Hiểu khái niệm **đường chéo** trái-trên → phải-dưới và đại lượng bất biến `i - j`.
3. Đi từ cách "gom theo đường chéo" (dict) → "so với ô trái-trên" → "so hai hàng liền kề bằng slicing".
4. Trả lời câu hỏi follow-up: chỉ được nạp **một hàng** vào bộ nhớ thì làm sao?
5. Trace, chứng minh đúng, phân tích Time/Space và luyện biến thể (anti-diagonal, Hankel).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — mọi đường chéo có cùng giá trị?

Cho ma trận `m x n`. Trả về `True` nếu ma trận là **Toeplitz**, ngược lại `False`.

Ma trận Toeplitz: **mọi đường chéo** đi từ trái-trên xuống phải-dưới đều gồm các phần tử **bằng nhau**.

**Ví dụ 1:**
```text
1 2 3 4
5 1 2 3
9 5 1 2
```
Các đường chéo: `[9]`, `[5,5]`, `[1,1,1]`, `[2,2,2]`, `[3,3]`, `[4]` → `True`.

**Ví dụ 2:**
```text
1 2
2 2
```
Đường chéo `[1, 2]` khác nhau → `False`.

**Constraints:** `1 <= m, n <= 20`, `0 <= matrix[i][j] <= 99`.

**Follow up:**
- Nếu ma trận nằm trên đĩa và bộ nhớ chỉ chứa được **một hàng** mỗi lần?
- Nếu ma trận lớn đến mức chỉ nạp được **một phần hàng**?

In [1]:
matrix1 = [[1, 2, 3, 4],
           [5, 1, 2, 3],
           [9, 5, 1, 2]]
matrix2 = [[1, 2],
           [2, 2]]
for row in matrix1:
    print(row)

[1, 2, 3, 4]
[5, 1, 2, 3]
[9, 5, 1, 2]


## 2. Nền tảng Python: ma trận là list của các list

- `matrix[i]` là **hàng** thứ `i` (một list).
- `matrix[i][j]` là phần tử ở hàng `i`, cột `j` (đều đếm từ 0).
- `len(matrix)` = số hàng `m`; `len(matrix[0])` = số cột `n`.

In [2]:
m, n = len(matrix1), len(matrix1[0])
print("m =", m, "hàng; n =", n, "cột")
print("hàng 1:", matrix1[1])
print("matrix1[1][2] =", matrix1[1][2])   # hàng 1, cột 2
print("ô cuối:", matrix1[-1][-1])           # -1 = phần tử cuối

m = 3 hàng; n = 4 cột
hàng 1: [5, 1, 2, 3]
matrix1[1][2] = 2
ô cuối: 2


### 2.1. Hai vòng `for` lồng nhau để duyệt mọi ô

Vòng ngoài chạy hàng `i`, vòng trong chạy cột `j`. Tổng cộng `m · n` lần.

In [3]:
for i in range(len(matrix2)):
    for j in range(len(matrix2[0])):
        print(f"(i={i}, j={j}) -> {matrix2[i][j]}")

(i=0, j=0) -> 1
(i=0, j=1) -> 2
(i=1, j=0) -> 2
(i=1, j=1) -> 2


## 3. Nền tảng tư duy: đường chéo và đại lượng `i - j`

Đi dọc một đường chéo trái-trên → phải-dưới nghĩa là `i` tăng 1 **và** `j` tăng 1. Khi đó `i - j` **không đổi**.

→ Mỗi đường chéo được đánh dấu bởi một số `d = i - j`, chạy từ `-(n-1)` đến `m-1`. Với `m=3, n=4` có `3 + 4 - 1 = 6` đường chéo.

Bảng `i - j` của ví dụ 1:
```text
 0 -1 -2 -3
 1  0 -1 -2
 2  1  0 -1
```
So với ma trận: các ô có cùng `i-j` đều cùng giá trị!

In [4]:
for i in range(3):
    print([i - j for j in range(4)])

[0, -1, -2, -3]
[1, 0, -1, -2]
[2, 1, 0, -1]


## 4. Cách 1 — Trực giác: gom từng đường chéo bằng `dict`

Ý tưởng: dùng **dictionary** `first[d]` lưu giá trị **đầu tiên** gặp trên đường chéo `d = i - j`. Gặp ô khác cùng đường chéo thì so với giá trị đã lưu.

Ôn `dict`:
- `first = {}` tạo dict rỗng.
- `d in first` kiểm tra key đã có chưa (trung bình `O(1)`).
- `first[d] = v` thêm/ghi; `first[d]` đọc.

In [5]:
first = {}
first[0] = 1
first[-1] = 2
print(first)
print(0 in first, 5 in first, first[-1])

{0: 1, -1: 2}
True False 2


In [6]:
def is_toeplitz_dict(matrix: list[list[int]]) -> bool:
    first = {}                          # d = i - j  ->  giá trị đầu tiên của đường chéo d
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            d = i - j
            if d not in first:
                first[d] = matrix[i][j]
            elif first[d] != matrix[i][j]:
                return False
    return True

print(is_toeplitz_dict(matrix1))   # True
print(is_toeplitz_dict(matrix2))   # False

True
False


### Code nộp LeetCode — Cách 1 (dict theo đường chéo)

In [7]:
class Solution:
    def isToeplitzMatrix(self, matrix: list[list[int]]) -> bool:
        first = {}
        for i, row in enumerate(matrix):
            for j, val in enumerate(row):
                if i - j not in first:
                    first[i - j] = val
                elif first[i - j] != val:
                    return False
        return True

print(Solution().isToeplitzMatrix(matrix1), Solution().isToeplitzMatrix(matrix2))

True False


### `enumerate` — vừa lấy chỉ số vừa lấy giá trị

`for i, row in enumerate(matrix)` cho từng cặp `(i, row)`. Gọn hơn `for i in range(len(matrix)): row = matrix[i]`.

In [8]:
for i, row in enumerate(matrix2):
    for j, val in enumerate(row):
        print(i, j, val)

0 0 1
0 1 2
1 0 2
1 1 2


**Độ phức tạp cách 1:** Time `O(m·n)`; Space `O(m + n)` cho dict (có `m + n - 1` đường chéo).

## 5. Nút thắt và insight — chỉ cần so với ô **trái-trên**

Dict tốn thêm bộ nhớ. Nhận xét: trên một đường chéo, nếu **mỗi ô bằng ô ngay trước nó** (ô trái-trên `matrix[i-1][j-1]`) thì theo bắc cầu cả đường chéo bằng nhau.

```text
a = b, b = c, c = d   ⇒   a = b = c = d
```

→ Chỉ cần kiểm tra: với mọi `i >= 1, j >= 1`: `matrix[i][j] == matrix[i-1][j-1]`. Không cần bộ nhớ thêm.

Ô ở hàng 0 hoặc cột 0 là **điểm đầu** của đường chéo nên không có ô trái-trên — bắt đầu từ `1`.

In [9]:
i, j = 2, 3
print(f"matrix1[{i}][{j}] = {matrix1[i][j]}, ô trái-trên matrix1[{i-1}][{j-1}] = {matrix1[i-1][j-1]}")

matrix1[2][3] = 2, ô trái-trên matrix1[1][2] = 2


## 6. Lời giải chính — so với ô trái-trên

Giảng từng đoạn:
```python
for i in range(1, len(matrix)):          # (1) bỏ hàng 0
    for j in range(1, len(matrix[0])):   # (2) bỏ cột 0
        if matrix[i][j] != matrix[i-1][j-1]:   # (3) khác ô trái-trên?
            return False                        #     -> phá vỡ một đường chéo
return True                                     # (4) không có vi phạm nào
```
- `(1)(2)` bắt đầu từ 1 để `i-1`, `j-1` không âm (tránh lấy nhầm `matrix[-1]` = hàng cuối!).
- `(3)` một cặp khác nhau là đủ kết luận `False` → dừng sớm.

In [10]:
def is_toeplitz(matrix: list[list[int]]) -> bool:
    for i in range(1, len(matrix)):
        for j in range(1, len(matrix[0])):
            if matrix[i][j] != matrix[i - 1][j - 1]:
                return False
    return True

print(is_toeplitz(matrix1), is_toeplitz(matrix2))

True False


## 7. TRACE từng bước

Ví dụ 1 có `(m-1)(n-1) = 2·3 = 6` phép so sánh:

| i | j | matrix[i][j] | matrix[i-1][j-1] | Bằng? |
|---|---|---|---|---|
| 1 | 1 | 1 | 1 | ✔ |
| 1 | 2 | 2 | 2 | ✔ |
| 1 | 3 | 3 | 3 | ✔ |
| 2 | 1 | 5 | 5 | ✔ |
| 2 | 2 | 1 | 1 | ✔ |
| 2 | 3 | 2 | 2 | ✔ |

Ví dụ 2 dừng ngay ở `(1,1)`: `2 != 1`.

In [11]:
def trace_is_toeplitz(matrix):
    print("ma trận:", matrix)
    for i in range(1, len(matrix)):
        for j in range(1, len(matrix[0])):
            cur, up_left = matrix[i][j], matrix[i - 1][j - 1]
            same = cur == up_left
            print(f"  (i={i}, j={j}) cur={cur} | trái-trên=({i-1},{j-1})={up_left} | bằng? {same}")
            if not same:
                print("  -> vi phạm, trả False")
                return False
    print("  -> không có vi phạm, trả True")
    return True

_ = trace_is_toeplitz(matrix1)
_ = trace_is_toeplitz(matrix2)
_ = trace_is_toeplitz([[1, 2, 3], [4, 1, 9]])

ma trận: [[1, 2, 3, 4], [5, 1, 2, 3], [9, 5, 1, 2]]
  (i=1, j=1) cur=1 | trái-trên=(0,0)=1 | bằng? True
  (i=1, j=2) cur=2 | trái-trên=(0,1)=2 | bằng? True
  (i=1, j=3) cur=3 | trái-trên=(0,2)=3 | bằng? True
  (i=2, j=1) cur=5 | trái-trên=(1,0)=5 | bằng? True
  (i=2, j=2) cur=1 | trái-trên=(1,1)=1 | bằng? True
  (i=2, j=3) cur=2 | trái-trên=(1,2)=2 | bằng? True
  -> không có vi phạm, trả True
ma trận: [[1, 2], [2, 2]]
  (i=1, j=1) cur=2 | trái-trên=(0,0)=1 | bằng? False
  -> vi phạm, trả False
ma trận: [[1, 2, 3], [4, 1, 9]]
  (i=1, j=1) cur=1 | trái-trên=(0,0)=1 | bằng? True
  (i=1, j=2) cur=9 | trái-trên=(0,1)=2 | bằng? False
  -> vi phạm, trả False


## 8. Code nộp LeetCode — lời giải chính

In [12]:
class Solution:
    def isToeplitzMatrix(self, matrix: list[list[int]]) -> bool:
        for i in range(1, len(matrix)):
            for j in range(1, len(matrix[0])):
                if matrix[i][j] != matrix[i - 1][j - 1]:
                    return False
        return True

sol = Solution()
assert sol.isToeplitzMatrix(matrix1) is True
assert sol.isToeplitzMatrix(matrix2) is False
print("Solution OK")

Solution OK


## 9. Cách 3 — So hai hàng liền kề bằng slicing (trả lời follow-up 1)

Hàng `i+1` của ma trận Toeplitz chính là hàng `i` **dịch phải một ô** (thêm một phần tử mới ở đầu):
```text
hàng 0:    1 2 3 4
hàng 1:  5 1 2 3        -> hàng1[1:] == hàng0[:-1]
```
Slicing:
- `row[:-1]` = mọi phần tử trừ phần tử cuối.
- `row[1:]` = mọi phần tử trừ phần tử đầu.

Follow-up "chỉ nạp được một hàng": ta chỉ cần **giữ hàng trước** và đọc hàng hiện tại (thật ra là 2 hàng, hoặc giữ `n-1` phần tử của hàng trước) → bộ nhớ `O(n)` thay vì `O(m·n)`.

In [13]:
r0, r1 = matrix1[0], matrix1[1]
print("r0[:-1] =", r0[:-1])
print("r1[1:]  =", r1[1:])
print("bằng nhau?", r0[:-1] == r1[1:])

r0[:-1] = [1, 2, 3]
r1[1:]  = [1, 2, 3]
bằng nhau? True


In [14]:
def is_toeplitz_rows(matrix: list[list[int]]) -> bool:
    for i in range(len(matrix) - 1):
        if matrix[i][:-1] != matrix[i + 1][1:]:
            return False
    return True

def is_toeplitz_stream(rows) -> bool:
    # rows có thể là một iterator đọc từng hàng từ đĩa: ta chỉ giữ hàng trước
    prev = None
    for row in rows:
        if prev is not None and prev[:-1] != row[1:]:
            return False
        prev = row
    return True

print(is_toeplitz_rows(matrix1), is_toeplitz_rows(matrix2))
print(is_toeplitz_stream(iter(matrix1)), is_toeplitz_stream(iter(matrix2)))

True False
True False


In [15]:
class Solution:
    def isToeplitzMatrix(self, matrix: list[list[int]]) -> bool:
        return all(matrix[i][:-1] == matrix[i + 1][1:] for i in range(len(matrix) - 1))

print(Solution().isToeplitzMatrix(matrix1), Solution().isToeplitzMatrix(matrix2))

True False


### `all(...)` — đúng khi mọi phần tử đều đúng

`all(iterable)` trả `True` nếu không có phần tử nào sai (và dừng ngay khi gặp phần tử sai). `all([])` = `True` — nên ma trận 1 hàng tự động là Toeplitz.

In [16]:
print(all([True, True]), all([True, False]), all([]))

True False True


### Follow-up 2: chỉ nạp được **một phần hàng**

Chia mỗi hàng thành các **khúc** (chunk) theo cột. Với khúc cột `[c, c+k)` của hàng `i+1`, ta cần khúc cột `[c-1, c+k-1)` của hàng `i` — tức là chỉ cần **một phần tử chồng lấn** ở biên khúc. Đọc lần lượt từng khúc của hai hàng liền kề và so sánh; bộ nhớ `O(k)`.

Cell dưới mô phỏng việc so sánh từng khúc kích thước `k`.

In [17]:
def is_toeplitz_chunks(matrix, k=2):
    m, n = len(matrix), len(matrix[0])
    for i in range(m - 1):
        for start in range(1, n, k):            # khúc cột [start, start+k) của hàng i+1
            end = min(n, start + k)
            lower = matrix[i + 1][start:end]     # "nạp" một phần hàng i+1
            upper = matrix[i][start - 1:end - 1] # phần tương ứng của hàng i (lệch 1 cột)
            if lower != upper:
                return False
    return True

for M in [matrix1, matrix2, [[1, 2, 3, 4, 5], [0, 1, 2, 3, 4], [7, 0, 1, 2, 9]]]:
    print(is_toeplitz_chunks(M, k=2), is_toeplitz(M))

True True
False False
False False


## 10. Vì sao đúng?

**Mệnh đề:** Ma trận Toeplitz ⇔ với mọi `i >= 1, j >= 1`: `matrix[i][j] == matrix[i-1][j-1]`.

- (⇒) Nếu Toeplitz, `(i,j)` và `(i-1,j-1)` cùng đường chéo (`i-j` bằng nhau) nên bằng nhau.
- (⇐) Lấy hai ô bất kỳ cùng đường chéo `(i,j)` và `(i+t, j+t)`. Chuỗi `(i,j) → (i+1,j+1) → ... → (i+t,j+t)` mỗi bước là một cặp "ô và ô trái-trên" đã kiểm tra bằng nhau ⇒ bắc cầu, hai ô bằng nhau.

**Invariant vòng lặp:** khi đang xét `(i, j)`, mọi cặp đã xét trước đó đều bằng nhau; nếu gặp cặp khác nhau thì chắc chắn có một đường chéo không đồng nhất → trả `False` là đúng. Duyệt hết không vi phạm ⇒ `True`.

Cách 3 (hàng) chỉ là gom các so sánh `matrix[i+1][j] == matrix[i][j-1]` của cùng một hàng thành một phép so sánh list.

## 11. Độ phức tạp và so sánh

Gọi `m` = số hàng, `n` = số cột.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Dict theo `i - j` | `O(m·n)` | `O(m+n)` | Trực quan "gom đường chéo" |
| 2. So với ô trái-trên | `O(m·n)` | `O(1)` | **Khuyến nghị**, dừng sớm |
| 3. So hai hàng (slicing) | `O(m·n)` | `O(n)` (slice tạo list mới) | Gọn, hợp follow-up 1 |
| 3b. Theo khúc | `O(m·n)` | `O(k)` | Follow-up 2 |

## 12. Bộ test

1. Ví dụ đề (True / False).
2. Biên: ma trận `1x1`, một hàng, một cột (luôn True).
3. Vi phạm ở **ô cuối cùng** (để chắc vòng lặp chạy tới cuối).
4. Phản ví dụ cho lỗi "so với ô phải-trên" (nhầm hướng chéo).
5. Đối chiếu 4 cách trên ma trận ngẫu nhiên (cả Toeplitz được sinh có chủ đích).

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", is_toeplitz(matrix1), True)
check("đề 2", is_toeplitz(matrix2), False)
check("1x1", is_toeplitz([[7]]), True)
check("một hàng", is_toeplitz([[1, 2, 3, 4]]), True)
check("một cột", is_toeplitz([[1], [2], [3]]), True)
check("sai ở ô cuối", is_toeplitz([[1, 2, 3], [4, 1, 2], [5, 4, 9]]), False)
check("anti-diagonal không phải Toeplitz", is_toeplitz([[1, 2], [2, 3]]), False)

PASS đề 1: True
PASS đề 2: False
PASS 1x1: True
PASS một hàng: True
PASS một cột: True
PASS sai ở ô cuối: False
PASS anti-diagonal không phải Toeplitz: False


In [19]:
import random
random.seed(766)

def random_toeplitz(m, n):
    # sinh giá trị cho mỗi đường chéo d = i - j rồi điền vào
    diag = {d: random.randint(0, 99) for d in range(-(n - 1), m)}
    return [[diag[i - j] for j in range(n)] for i in range(m)]

for _ in range(300):
    m, n = random.randint(1, 6), random.randint(1, 6)
    M = random_toeplitz(m, n)
    if random.random() < 0.5:                 # phá một ô ngẫu nhiên
        i, j = random.randrange(m), random.randrange(n)
        M[i][j] = (M[i][j] + random.randint(1, 5)) % 100
    expected = is_toeplitz_dict(M)
    assert is_toeplitz(M) == expected
    assert is_toeplitz_rows(M) == expected
    assert is_toeplitz_chunks(M, k=random.randint(1, 3)) == expected
    assert Solution().isToeplitzMatrix(M) == expected
print("300 ma trận ngẫu nhiên: 4 cách khớp nhau")

300 ma trận ngẫu nhiên: 4 cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1: Bắt đầu vòng lặp từ 0
```python
for i in range(len(matrix)):
    for j in range(len(matrix[0])):
        if matrix[i][j] != matrix[i-1][j-1]: ...   # SAI khi i=0 hoặc j=0
```
Python **không báo lỗi**: `matrix[-1]` là hàng cuối! Kết quả sai lặng lẽ.

### Lỗi 2: So với ô sai hướng (phải-trên `matrix[i-1][j+1]`)
Đó là đường chéo **ngược** (anti-diagonal), đề hỏi đường chéo trái-trên → phải-dưới.

### Lỗi 3: Slicing ngược chiều
`matrix[i][1:] == matrix[i+1][:-1]` — sai chiều dịch.

### Lỗi 4: Dùng `len(matrix)` cho số cột
Ma trận không vuông thì `n = len(matrix[0])`, không phải `len(matrix)`.

In [20]:
def wrong_from_zero(matrix):
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            if matrix[i][j] != matrix[i - 1][j - 1]:
                return False
    return True

def wrong_slice_direction(matrix):
    return all(matrix[i][1:] == matrix[i + 1][:-1] for i in range(len(matrix) - 1))

print("matrix1 Toeplitz nhưng wrong_from_zero ->", wrong_from_zero(matrix1))
M = [[1, 2], [2, 3]]
print("[[1,2],[2,3]]: wrong_slice_direction ->", wrong_slice_direction(M), "| đúng ->", is_toeplitz(M))

matrix1 Toeplitz nhưng wrong_from_zero -> False
[[1,2],[2,3]]: wrong_slice_direction -> True | đúng -> False


## 14. Mẹo nhận dạng pattern

- "Đường chéo trái-trên → phải-dưới" ⇒ đại lượng `i - j` không đổi. "Đường chéo ngược" ⇒ `i + j` không đổi.
- "Mọi phần tử trong nhóm bằng nhau" ⇒ chỉ cần so **phần tử liền kề** trong nhóm (bắc cầu), không cần so mọi cặp.
- Ma trận có "cấu trúc dịch" (hàng sau = hàng trước dịch một ô) ⇒ so sánh bằng slicing `[:-1]` vs `[1:]`.
- Bộ nhớ giới hạn / dữ liệu streaming ⇒ chỉ giữ **trạng thái tối thiểu** (hàng trước).

### Biến thể

1. **Ma trận Hankel**: mọi đường chéo **ngược** (`i + j` không đổi) bằng nhau → so `matrix[i][j]` với `matrix[i-1][j+1]`.
2. **Liệt kê các đường chéo** theo thứ tự (gom theo `i - j`).
3. **Diagonal Traverse (LeetCode 498)**, **Sort the Matrix Diagonally (LeetCode 1329)** dùng cùng ý tưởng nhóm theo `i - j`.
4. **Đếm số đường chéo không đồng nhất** thay vì trả True/False.

In [21]:
# Biến thể 2: liệt kê đường chéo theo d = i - j (từ trái-dưới lên phải-trên như đề)
from collections import defaultdict
groups = defaultdict(list)          # dict tự tạo list rỗng khi key chưa có
for i, row in enumerate(matrix1):
    for j, val in enumerate(row):
        groups[i - j].append(val)
for d in sorted(groups, reverse=True):
    print(f"d = {d:>2}: {groups[d]}")

d =  2: [9]
d =  1: [5, 5]
d =  0: [1, 1, 1]
d = -1: [2, 2, 2]
d = -2: [3, 3]
d = -3: [4]


`defaultdict(list)` (trong module chuẩn `collections`) giống `dict`, nhưng khi đọc một key chưa tồn tại nó tự tạo `list()` rỗng — nhờ đó `groups[d].append(val)` không cần kiểm tra `if d not in groups`.

## 15. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (tính tay):** Ma trận `[[3, 7, 0], [1, 3, 7], [8, 1, 3], [2, 8, 2]]` có Toeplitz không? Nếu không, ô `(i, j)` **đầu tiên** (theo thứ tự hàng rồi cột) bị vi phạm là ô nào? Gán vào `answer_ex1 = (i, j)`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `is_toeplitz_exercise` dùng so sánh ô trái-trên.

**Bài 3 (biến thể):** Viết `is_hankel(matrix)`: mọi đường chéo ngược (`i + j` không đổi) bằng nhau.

**Bài 4 (đếm):** Viết `count_bad_diagonals(matrix)` trả về số đường chéo (theo `i - j`) **không** đồng nhất.

**Bài 5 (mini challenge):** Viết `make_toeplitz(first_row, first_col)` dựng ma trận Toeplitz từ hàng đầu và cột đầu (giả sử `first_row[0] == first_col[0]`).

In [22]:
answer_ex1 = None   # TODO: (i, j)

def is_toeplitz_exercise(matrix: list[list[int]]) -> bool:
    # TODO 1: duyệt i từ 1, j từ 1
    # TODO 2: nếu khác ô trái-trên -> False
    pass

def is_hankel(matrix: list[list[int]]) -> bool:
    # TODO
    pass

# assert is_toeplitz_exercise([[1, 2], [2, 2]]) == False
# assert is_hankel([[1, 2, 3], [2, 3, 4]]) == True

In [23]:
def count_bad_diagonals(matrix: list[list[int]]) -> int:
    # TODO: gom theo i - j, đếm nhóm có nhiều hơn 1 giá trị khác nhau
    pass

def make_toeplitz(first_row: list[int], first_col: list[int]) -> list[list[int]]:
    # TODO
    pass

# assert count_bad_diagonals([[1, 2], [2, 2]]) == 1
# assert make_toeplitz([1, 2, 3], [1, 5]) == [[1, 2, 3], [5, 1, 2]]

### Gợi ý 3 mức

**Bài 1** — (1) Viết bảng `i - j`. (2) Kiểm tra từng ô so với ô trái-trên theo thứ tự. (3) Hàng 3: `matrix[3][2] = 2` so với `matrix[2][1] = 1`.

**Bài 2** — (1) Bắt đầu từ 1. (2) `range(1, len(matrix))`, `range(1, len(matrix[0]))`. (3) `if matrix[i][j] != matrix[i-1][j-1]: return False`.

**Bài 3** — (1) Đường chéo ngược: `i` tăng thì `j` giảm. (2) So với ô **phải-trên** `matrix[i-1][j+1]`. (3) `j` chạy tới `n - 2`.

**Bài 4** — (1) Dùng `defaultdict(set)`. (2) `groups[i-j].add(val)`. (3) Đếm nhóm có `len(set) > 1`.

**Bài 5** — (1) Giá trị ô chỉ phụ thuộc `i - j`. (2) `i - j >= 0` → lấy `first_col[i - j]`; ngược lại `first_row[j - i]`. (3) List comprehension 2 tầng.

---

## 16. Đáp án tham khảo

In [24]:
answer_ex1_solution = (3, 2)

def is_toeplitz_solution(matrix):
    for i in range(1, len(matrix)):
        for j in range(1, len(matrix[0])):
            if matrix[i][j] != matrix[i - 1][j - 1]:
                return False
    return True

def is_hankel_solution(matrix):
    for i in range(1, len(matrix)):
        for j in range(len(matrix[0]) - 1):
            if matrix[i][j] != matrix[i - 1][j + 1]:
                return False
    return True

def count_bad_diagonals_solution(matrix):
    groups = defaultdict(set)
    for i, row in enumerate(matrix):
        for j, val in enumerate(row):
            groups[i - j].add(val)
    return sum(len(vals) > 1 for vals in groups.values())

def make_toeplitz_solution(first_row, first_col):
    m, n = len(first_col), len(first_row)
    return [[first_col[i - j] if i >= j else first_row[j - i] for j in range(n)] for i in range(m)]

# Kiểm tra bài 1 bằng trace: ô vi phạm đầu tiên
ex = [[3, 7, 0], [1, 3, 7], [8, 1, 3], [2, 8, 2]]
first_bad = next(((i, j) for i in range(1, 4) for j in range(1, 3) if ex[i][j] != ex[i-1][j-1]), None)
assert first_bad == answer_ex1_solution
assert is_toeplitz_solution([[1, 2], [2, 2]]) is False and is_toeplitz_solution(matrix1) is True
assert is_hankel_solution([[1, 2, 3], [2, 3, 4]]) is True
assert is_hankel_solution([[1, 2], [3, 1]]) is False
assert count_bad_diagonals_solution([[1, 2], [2, 2]]) == 1
assert count_bad_diagonals_solution(matrix1) == 0
assert make_toeplitz_solution([1, 2, 3], [1, 5]) == [[1, 2, 3], [5, 1, 2]]
assert is_toeplitz(make_toeplitz_solution([4, 0, 9, 9], [4, 1, 2]))
print("Đáp án PASS toàn bộ test!")

Đáp án PASS toàn bộ test!


## 17. Template nhớ nhanh và checklist submit

```python
class Solution:
    def isToeplitzMatrix(self, matrix: list[list[int]]) -> bool:
        for i in range(1, len(matrix)):
            for j in range(1, len(matrix[0])):
                if matrix[i][j] != matrix[i - 1][j - 1]:
                    return False
        return True
```

Ghi nhớ:
- Đường chéo chính: `i - j` hằng số; đường chéo ngược: `i + j` hằng số.
- Một dòng: `all(r1[:-1] == r2[1:] for r1, r2 in zip(matrix, matrix[1:]))`.

**Checklist trước khi submit:**
- [ ] Vòng lặp bắt đầu từ `1` (tránh `matrix[-1]`)?
- [ ] Dùng `len(matrix[0])` cho số cột?
- [ ] So sánh với ô **trái-trên** `[i-1][j-1]`, không phải phải-trên?
- [ ] Ma trận 1 hàng / 1 cột trả `True`?
- [ ] Biết trả lời follow-up: giữ hàng trước (`O(n)`), hoặc chia khúc (`O(k)`).